# 3_6 — Абляции DPI-Flow (P1, component-contribution)

Отдельный ноутбук абляций. Каждая 1:1 отключает один компонент вклада; обучение/оценка на объектном (leakage-free) фолде — тот же протокол, что P0. Метрики **по 3 состояниям** (liq/stab/nostab + worst).

Варианты: full, wo_conformal, gaussian_posterior, wo_ode, wo_monotone, wo_risk_softauc, wo_censored_nliq, miss_vs, miss_grainsize. Логика — в `liquefaction_ai.evaluation.ablation_study`.

In [1]:
import os, sys
REPO = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))
if not os.path.exists(os.path.join(REPO, 'src')):
    REPO = os.getcwd()
sys.path.insert(0, os.path.join(REPO, 'src'))
os.chdir(REPO)
import numpy as np, pandas as pd
TABLES = os.path.join(REPO, 'results', 'analysis_tables'); os.makedirs(TABLES, exist_ok=True)
PUB = os.path.join(REPO, 'results', 'tables'); os.makedirs(PUB, exist_ok=True)
FIGS = os.path.join(REPO, 'results', 'analysis_figs'); os.makedirs(FIGS, exist_ok=True)
print('REPO =', REPO)

REPO = /Users/nick/Desktop/projects/liquefaction-ai


In [2]:
import json, torch
from liquefaction_ai import load_population_artifact
from liquefaction_ai.evaluation.ablation_study import run_ablation_fold, aggregate_ablations
device = torch.device('cpu'); QUICK = False
pop, config = load_population_artifact('data/demo_run')
meta = pop['meta']
feat_names = json.load(open('data/demo_run/feature_names.json'))['static_feature_names']
from liquefaction_ai.evaluation.cross_validation import build_folds
folds = build_folds(meta, config, seed=42, loo=False, n_splits=5)
print('folds:', len(folds))

folds: 5


## Прогон абляций (по умолчанию фолд 0; для CI — по всем фолдам)

In [3]:
FOLDS = [0, 1, 2]   # ≥3 фолда для CI ключевых абляций (рецензент); [] → все
use_folds = FOLDS if FOLDS else range(len(folds))
rows = []
for k in use_folds:
    rows.append(run_ablation_fold(pop, config, folds[k], k, feat_names, device, quick=QUICK))
abl = pd.concat(rows, ignore_index=True)
abl.to_csv(os.path.join(TABLES, 'ablations.csv'), index=False)
abl[['fold','ablation','AUPRC','ECE','Coverage_90','Traj_RMSE_worst','Physics_Violation_Rate','N_liq_logMAE']].round(4)

,fold,ablation,AUPRC,ECE,Coverage_90,Traj_RMSE_worst,Physics_Violation_Rate,N_liq_logMAE
0,0,full,0.9619,0.1060,0.7905,0.1599,0.0065,0.2678
1,0,wo_conformal,0.9619,0.1060,0.6582,0.1599,0.0065,0.2678
2,0,gaussian_posterior,0.9594,0.1175,0.7617,0.1499,0.0000,0.2808
3,0,wo_ode,0.9770,0.1546,0.7885,0.0990,0.0000,0.6673
4,0,blackbox_cummax,0.9770,0.1546,0.7885,0.0990,0.0000,0.6673
5,0,blackbox_raw,0.9893,0.1245,0.8038,0.0827,0.3779,0.7055
6,0,wo_monotone,0.9807,0.1138,0.7916,0.1147,0.1433,0.2787
7,0,wo_risk_softauc,0.9820,0.0886,0.8135,0.1117,0.0000,0.2740
8,0,wo_censored_nliq,0.9782,0.0906,0.7613,0.1559,0.0065,0.2792
9,0,miss_vs,0.9692,0.1052,0.8374,0.1177,0.0000,0.2886


In [4]:
summary = aggregate_ablations(abl)
summary.to_csv(os.path.join(TABLES, 'ablations_summary.csv'), index=False)
summary[[c for c in ['ablation','n_folds','AUPRC_mean','ECE_mean','Traj_RMSE_worst_mean','Physics_Violation_Rate_mean','N_liq_logMAE_mean'] if c in summary.columns]]

,ablation,n_folds,AUPRC_mean,ECE_mean,Traj_RMSE_worst_mean,Physics_Violation_Rate_mean,N_liq_logMAE_mean
0,blackbox_cummax,3,0.9864,0.0788,0.0694,0.0000,0.6137
1,blackbox_raw,3,0.9902,0.0639,0.0692,0.2248,0.7383
2,full,3,0.9703,0.0797,0.1456,0.0036,0.3282
3,gaussian_posterior,3,0.9783,0.0753,0.1519,0.0014,0.3317
4,miss_grainsize,3,0.9828,0.0642,0.1434,0.0047,0.3179
5,miss_vs,3,0.9758,0.0755,0.1296,0.0014,0.3248
6,no_aux,3,0.9805,0.0828,0.1380,0.0014,0.3388
7,no_prefix,3,0.8651,0.1636,0.2016,0.0022,0.3614
8,wo_censored_nliq,3,0.9786,0.0775,0.1442,0.0036,0.3307
9,wo_conformal,3,0.9703,0.0797,0.1456,0.0036,0.3282


## Ablation figures (component contribution, English)

In [5]:
from liquefaction_ai.evaluation import ablation_bars
from IPython.display import display
display(ablation_bars(summary, metric='Traj_RMSE_worst', higher_better=False, out_dir=FIGS))
display(ablation_bars(summary, metric='AUPRC', higher_better=True, out_dir=FIGS))

<Figure size 720x790 with 1 Axes>

<Figure size 720x790 with 1 Axes>

## Prefix-length sensitivity (п.6)

Ре-материализуйте `data/demo_run` с разным `config.prefix_len` (ноутбуки 1_x), затем для каждого K запустите этот ноутбук с `only='full'` и сравните метрики онсета/траектории vs длина префикса.